In [1]:
import os
import gc
import numpy as np
import pandas as pd
import random
from random import random as rd
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import HistGradientBoostingRegressor


def set_seed(seed=2021):
    np.random.seed(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


set_seed()

train_path = "../input/ventilator-pressure-prediction/train.csv"
train_df = pd.read_csv(
    train_path,
    usecols=["R", "C", "u_in", "u_out", "time_step", "pressure", "id", "breath_id"],
    dtype={
        "R": "int8",
        "C": "int8",
        "u_in": "float32",
        "u_out": "int8",
        "time_step": "float32",
        "pressure": "float32",
        "id": "int16",
        "breath_id": "int32",
    },
)


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add engineered interaction features in‑place."""
    df["R_C"] = df["R"].astype(np.float32) * df["C"].astype(np.float32)
    df["u_in_sq"] = df["u_in"] ** 2
    df["time_step_sq"] = df["time_step"] ** 2
    df["u_in_u_out"] = df["u_in"] * df["u_out"]
    return df


train_df = add_features(train_df)

sorted_pressures = np.sort(train_df["pressure"].unique())
total_pressures_len = len(sorted_pressures)




In [2]:
def round_to_nearest(preds: np.ndarray) -> np.ndarray:
    """Round each prediction to the closest pressure value observed in training."""
    idx = np.searchsorted(sorted_pressures, preds, side="left")
    idx_right = np.clip(idx, 0, total_pressures_len - 1)
    idx_left = np.clip(idx_right - 1, 0, total_pressures_len - 1)

    left_vals = sorted_pressures[idx_left]
    right_vals = sorted_pressures[idx_right]

    use_left = np.abs(preds - left_vals) < np.abs(right_vals - preds)
    return np.where(use_left, left_vals, right_vals)




In [3]:
test_path = "../input/ventilator-pressure-prediction/test.csv"
sample_sub_path = "../input/ventilator-pressure-prediction/sample_submission.csv"

test_df = pd.read_csv(
    test_path,
    usecols=["R", "C", "u_in", "u_out", "time_step", "id", "breath_id"],
    dtype={
        "R": "int8",
        "C": "int8",
        "u_in": "float32",
        "u_out": "int8",
        "time_step": "float32",
        "id": "int16",
        "breath_id": "int32",
    },
)

test_df = add_features(test_df)

feature_cols = [
    "R",
    "C",
    "u_in",
    "u_out",
    "time_step",
    "R_C",
    "u_in_sq",
    "time_step_sq",
    "u_in_u_out",
]
target_col = "pressure"

X = train_df[feature_cols].to_numpy(dtype=np.float32, copy=False)
y = train_df[target_col].to_numpy(dtype=np.float32, copy=False)

del train_df
gc.collect()

# Enable early stopping to avoid unnecessary boosting rounds and reduce histogram bins for speed.
model = HistGradientBoostingRegressor(
    max_iter=5000,
    learning_rate=0.05,
    max_depth=8,
    max_bins=128,  # fewer bins → faster histogram construction, negligible impact
    random_state=42,
    loss="absolute_error",
    early_stopping=True,  # stop when validation loss ceases to improve
    validation_fraction=0.1,  # use 10 % of data for validation (default)
    n_iter_no_change=10,  # patience for early stopping
    verbose=0,
)

model.fit(X, y)

sample_idx = np.random.choice(len(y), size=min(200_000, len(y)), replace=False)
val_mae = mean_absolute_error(y[sample_idx], model.predict(X[sample_idx]))
print(f"Validation MAE on a sample: {val_mae:.5f}")

test_pred_raw = model.predict(
    test_df[feature_cols].to_numpy(dtype=np.float32, copy=False)
)
test_pred = round_to_nearest(
    test_pred_raw
)  # keep predictions on realistic pressure levels

submission = pd.read_csv(sample_sub_path)
submission["pressure"] = test_pred
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}")

Validation MAE on a sample: 2.04375
Submission saved to submission.csv
